# Output to ODS Demonstration

Just a quick thing of outputting the raw magnetics signals from the BP-LOMN probes in the SPARC device description.

1. Load probe information from SPARC device description
2. Set up and run simulation
3. Export as ODS file

In [ ]:
%load_ext autoreload
%autoreload 2

import json

from popsim import PACKAGE_ROOT

# TODO(ZanderKeith) can't add this dependency to the repo yet so for now I just copied it here in my local version
device_description_path = f"{PACKAGE_ROOT}/../submodules/Device-description/device_description/SPARC/240510/device.json"

with open(device_description_path) as f:
    device_description = json.load(f)

probe_details = [probe for probe in device_description["magnetics"]["b_field_pol_probe"] if "BP-LOMN" in probe["name"]]

R0 = device_description["summary"]["global_quantities"]["r0"]["value"]


In [ ]:
from popsim.modules.magnetic_diagnostics import BFieldPoloidalProbes, LowNArray, load_lown_config
from popsim.modules.tearing import Tearing
from popsim.simulate import SimInput, simulate
from popsim.simulators.tearing_sim.model import TearingSim

tearing_module, tearing_initial_state, tearing_params, time_base = Tearing.default_setup()
lown_array_module = LowNArray.default_setup(empty=True)

b_field_poloidal_config = BFieldPoloidalProbes.Config(
    func_Bp_per_A=load_lown_config()[1],
    probe_details=probe_details,
    R0=R0,
)

b_field_poloidal_probes_module = BFieldPoloidalProbes(config=b_field_poloidal_config)

sim_config = TearingSim.Config(
    tearing_module=tearing_module,
    lown_array_module=lown_array_module,
    b_field_poloidal_probes_module=b_field_poloidal_probes_module,
)

sim_initial_state = TearingSim.State(tearing_state=tearing_initial_state)

sim_params = TearingSim.Params(tearing_params=tearing_params)

tearing_sim = TearingSim(config=sim_config)


sim_input = SimInput(time=time_base, initial_state=sim_initial_state, params=sim_params)
sim_xarray = simulate(tearing_sim, sim_inputs=sim_input)
sim_xarray

In [ ]:
import holoviews as hv

hv.extension("matplotlib")

probe_data_plots = []
for probe in probe_details[:4]:
    probe_data = sim_xarray[f"output.b_field_poloidal_probes_out.Bp.{probe['name']}"]
    probe_data_plot = hv.Scatter((sim_xarray.time, probe_data), label=probe["name"])
    probe_data_plot.opts(xlabel="Time [s]", ylabel="Bp [T]")
    probe_data_plots.append(probe_data_plot)

hv.Layout(probe_data_plots).opts(title="B Field Poloidal Probes")


In [ ]:
from omas import ODS, save_omas_h5

ods = ODS(imas_version="3.40.0", consistency_check=True)

tearing_sim.add_to_ods(ods, sim_xarray)

save_omas_h5(ods, "test_tearing_out.h5")